In [1]:
#Importing necessary libraries
import ee
import geemap

In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project='nowmi-learn')

In [4]:
print(ee.String("Hello from GEE").getInfo())

Hello from GEE


In [5]:
#Using ROI from FAO/GAUL admin level 2 boundaries to stream down analysis
#example usage from GEE: ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")

roi_admin = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")
brisbane = roi_admin.filter(ee.Filter.eq('ADM2_NAME','Brisbane (C)'))


In [6]:
geometry = brisbane.geometry().buffer(500.00)

In [7]:
before_c = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(geometry)
    .filterDate('2021-11-01','2021-12-11')
    .filter(ee.Filter.eq('instrumentMode','IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .filter(ee.Filter.eq('resolution_meters',10))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
    .select('VH')).mosaic().clip(geometry)

In [8]:
after_c = (ee.ImageCollection('COPERNICUS/S1_GRD')
         .filterBounds(geometry)
         .filterDate('2022-02-23','2022-03-01')
         .filter(ee.Filter.eq('instrumentMode','IW'))
         .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
         .filter(ee.Filter.eq('resolution_meters',10))
         .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
         .select('VH')).mosaic().clip(geometry)

In [10]:
Map = geemap.Map(center=(-27.805, 152.77), zoom=10) #Note: geemap takes values as lat,long 
                                                                #not X,Y(long, lat)

In [15]:
Map.addLayer(before_c, {'min': -25, 'max': 0}, 'Before Flood')
Map.addLayer(after_c, {'min': -25, 'max': 0}, 'After Flood')

In [12]:
Map.addLayerControl()

In [13]:
Map

Map(center=[-27.805, 152.77], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

In [57]:
brisbane

In [23]:
# Speckle filtering - simple method is by applying a focal median of 30 with circle filter. 
# Right scientific approach is by using Refined Lee filter
smoothing_radius = 30
before_filtered = before_c.focal_median(smoothing_radius, 'circle', 'meters')
after_filtered = after_c.focal_median(smoothing_radius, 'circle', 'meters')


In [24]:
Map1 = geemap.Map(center=(-27.805, 152.77), zoom=9)

In [25]:
Map1.addLayer(before_c, {'min': -25, 'max': 0}, 'Before Flood')
Map1.addLayer(after_c, {'min': -25, 'max': 0}, 'After Flood')
Map1.addLayer(before_filtered, {'min': -25, 'max': 0}, 'Before Flood Filtered')
Map1.addLayer(after_filtered, {'min': -25, 'max': 0}, 'After Flood Filtered')

In [26]:
Map1.centerObject(geometry)

In [38]:
Map1

Map(bottom=2430983.0, center=[-27.505454677776644, 153.10332944394196], controls=(WidgetControl(options=['posi…

In [43]:
difference = after_filtered.subtract(before_filtered)
# since GRD is log scale subtraction makes sense...!?
threshold = -3
flooded = difference.gt(threshold).rename('water').selfMask()
Map1.addLayer(flooded, {min:0, max:1}, 'Initial Flood Area')